# Facility Capacity — GPU Training

**Goal:** predict `capacity` (beds/scale) for health facilities from the Virtue Foundation dataset.

**Before running:** in the top-right next to the compute dropdown, switch the environment from *Standard* to **AI | GPU** (A10/H100). Then *Run All*. The training cell checks for CUDA and warns if you're still on a CPU environment.

**Data:** `facilities` (10,088 rows; ~2,520 have a usable numeric `capacity` label). The NFHS-5 health indicators and pincode directory tables stay unloaded — join keys are documented in the last cell so they can be conjoined later without rework.

## Weights & Biases setup

Open this notebook from the project Git folder with `wandb_models.py` and `gpu_capacity_models.py` next to it.
Install the dependencies below. Each training section logs separate PyTorch MLP and
histogram boosting runs with test metrics and the exact fitted pipelines or weights.
The MLP run also records per-epoch training and calibration Huber loss, learning rate,
and the chosen checkpoint.
`RANDOM_STATE` seeds the data split, initial model weights, and batch order.
Runs are offline by default. Set `WANDB_MODE=online`, `WANDB_PROJECT`, and
`WANDB_ENTITY` for a shared project, and provide `WANDB_API_KEY` through a secret.


In [ ]:
%pip install "wandb==0.31.2" "skops==0.16.0" "safetensors==0.8.0"


In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

TARGET = "capacity"
RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
]
CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [0]:
# Load the facilities table and normalize the messy string columns.
# Many columns are strings holding numbers or literal 'null'/'None' text, so we
# convert them to proper NaN/numeric before anything else.
NULL_STRINGS = {"null", "none", "nan", "n/a", "na", "", "-", "unknown"}

def to_na(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().where(
        lambda s: ~s.isin(NULL_STRINGS), other=pd.NA
    )

raw = spark.table(TBL_FACILITIES).toPandas()

facilities = raw.copy()
for col in [TARGET] + NUMERIC_FEATURES:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce").astype("float64")
for col in CATEGORICAL_FEATURES:
    category = to_na(facilities[col])
    facilities[col] = category.astype("object").where(category.notna(), np.nan)

# Free-text signal (verified by screening on the 2,517 labeled rows):
#   equipment/capability/procedure/specialties string lengths correlate with capacity
#   (Spearman ~0.33 / 0.38 / 0.23 / 0.15); affiliationTypeIds presence is strong
#   (median 200 when present vs 52 when absent) though only ~1,249 rows have it.
#   description length showed no signal (~0.02) and is left out.
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0).astype("float64")
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

# Keep only rows with a usable target label.
labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"facilities rows: {len(facilities)} | rows with a capacity label: {len(labeled)}")
print(f"Dropped {len(facilities) - len(labeled)} rows without a capacity value")
display(labeled[TARGET].describe().to_frame())

In [0]:
# EDA: target distribution and feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(labeled[TARGET], bins=50)
axes[0].set_title("Capacity distribution (right-skewed -> log1p helps)")
axes[0].set_xlabel("capacity")

axes[1].hist(np.log1p(labeled[TARGET]), bins=50)
axes[1].set_title("log1p(capacity)")
plt.tight_layout()
plt.show()

corr = labeled[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
corr.plot(kind="barh", ax=ax)
ax.set_title("Correlation of numeric features with capacity")
plt.tight_layout()
plt.show()

cap_by_type = labeled.groupby("facilityTypeId")[TARGET].agg(["count", "median"])
display(cap_by_type)
print(corr)

In [0]:
# Split: no event/observation timestamp exists and rows are independent
# facilities, so a shuffled random split is appropriate (80/10/10).
from sklearn.model_selection import train_test_split

X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Preprocessing fit on the training split only.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train).astype(np.float32)
X_val_prep = preprocessor.transform(X_val).astype(np.float32)
X_test_prep = preprocessor.transform(X_test).astype(np.float32)
print(f"feature matrix shape after preprocessing: {X_train_prep.shape}")

In [0]:
# Fit a histogram-boosting baseline on the same training features.
from gpu_capacity_models import fit_capacity_histogram

baseline = fit_capacity_histogram(preprocessor, X_train_prep, y_train, seed=RANDOM_STATE)


def regression_report(y_true, y_pred, name):
    error = np.asarray(y_pred) - y_true.to_numpy()
    absolute_error = np.abs(error)
    metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "median_ae": np.median(absolute_error),
        "p90_ae": np.quantile(absolute_error, 0.9),
        "mean_bias": error.mean(),
    }
    print(f"{name}: MAE {metrics['mae']:.2f} | RMSE {metrics['rmse']:.2f} | R2 {metrics['r2']:.3f}")
    return metrics


baseline_calibration_metrics = regression_report(y_val, baseline.predict(X_val), "Histogram boosting calibration")
baseline_metrics = regression_report(y_test, baseline.predict(X_test), "Histogram boosting test")


In [0]:
# PyTorch MLP with Huber loss on log1p(bed capacity).
from gpu_capacity_models import fit_capacity_mlp

mlp_fit = fit_capacity_mlp(
    X_train_prep, y_train, X_val_prep, y_val,
    seed=RANDOM_STATE, max_epochs=300, patience=30,
)
mlp = mlp_fit["model"]
device = mlp_fit["device"]


In [0]:
# Evaluate the selected MLP checkpoint in bed units.
def predict_beds(features):
    mlp.eval()
    with torch.inference_mode():
        matrix = torch.as_tensor(features, dtype=torch.float32, device=device)
        return np.expm1(mlp(matrix).cpu().numpy().ravel())


mlp_calibration_metrics = regression_report(y_val, predict_beds(X_val_prep), "MLP calibration")
mlp_pred = predict_beds(X_test_prep)
mlp_metrics = regression_report(y_test, mlp_pred, "MLP test")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, mlp_pred, s=12, alpha=0.5)
ax.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1)
ax.set_xlabel("actual capacity")
ax.set_ylabel("predicted capacity")
ax.set_title("Test set: predicted vs actual")
plt.tight_layout()
plt.show()


In [0]:
# Log both fitted models and aggregate evaluation in Weights & Biases.
from wandb_models import log_sklearn_capacity, log_torch_capacity

shared_config = {
    "variant": "base",
    "source_tables": [TBL_FACILITIES],
    "text_features": TEXT_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "split_unit": "source_row",
    "split_seed": RANDOM_STATE,
    "n_train": len(X_train),
    "n_validation": len(X_val),
    "n_test": len(X_test),
    "target_field": TARGET,
    "target_unit": "beds",
}

capacity_run = log_torch_capacity(
    name="capacity-mlp-base",
    model=mlp,
    preprocessor=preprocessor,
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    metrics=mlp_metrics,
    baseline_metrics=baseline_metrics,
    config={**shared_config, **mlp_fit["config"],
            "prediction_postprocess": "Apply expm1 once to the MLP output to get beds."},
    history=mlp_fit["history"],
    evaluation_tables={"heldout_by_split": [
        {"split": "calibration", "model": "mlp", **mlp_calibration_metrics},
        {"split": "test", "model": "mlp", **mlp_metrics},
        {"split": "calibration", "model": "histogram_boosting", **baseline_calibration_metrics},
        {"split": "test", "model": "histogram_boosting", **baseline_metrics},
    ]},
)

boosting_estimator = baseline.named_steps["histogram_boosting"].regressor_
baseline_run = log_sklearn_capacity(
    name="capacity-hist-gradient-boosting-base",
    model=baseline,
    metrics=baseline_metrics,
    config={**shared_config,
            "device": "cpu",
            "device_name": "CPU",
            "estimator": "HistGradientBoostingRegressor",
            "estimator_parameters": boosting_estimator.get_params(),
            "boosting_iterations": boosting_estimator.n_iter_,
            "prediction_postprocess": "The fitted pipeline applies expm1 and predict() returns beds."},
    target_transform={"training": "log1p", "prediction": "expm1 within pipeline.predict"},
    evaluation_tables={"heldout_by_split": [
        {"split": "calibration", **baseline_calibration_metrics},
        {"split": "test", **baseline_metrics},
    ]},
)
print("MLP W&B run", capacity_run)
print("Histogram boosting W&B run", baseline_run)


## Joining the other two tables later

The NFHS-5 and pincode tables are intentionally left out of training, but the keys to conjoin are:

* **facilities → pincode directory:** `address_zipOrPostcode` (facilities, string) ↔ `pincode` (directory, bigint). Watch state consistency: `address_stateOrRegion` ↔ `statename`, `address_city` ↔ `district`.
* **facilities → NFHS-5:** via the pincode directory (`district`/`statename` ↔ `district_name`/`state_ut`), or approximately via `latitude`/`longitude` to the nearest district centroid.
* Pincode `latitude`/`longitude` are strings — parse with `pd.to_numeric(errors="coerce")`.

## Next steps

* Compare the test metrics against the logged W&B gradient boosting baseline.
* If capacity labels with text values exist beyond the numeric ones, extend the cleaning cell to parse them (e.g., "100 beds").
* Feature ideas: TF-IDF on `description`/`specialties`, richer handling of `yearEstablished`, county-level aggregates from the pincode directory.